# CSC 786 — NIDS data collection

This notebook validates two public network-flow datasets, saves reproducible samples, and collects one CVE record from the NVD API. The API data are kept separate from the network-flow data.

## 1. Setup

Download the two original CSVs from the official sources linked in `README.md`. Put them either in `data/raw/` or a folder in your own Google Drive.

When prompted, enter the folder path relative to `My Drive`, or press Enter to use `data/raw/`. The dataset files are not included in this repository.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display

REPO_NAME = 'csc786-ethical-data-workflow'
REPO_URL = 'https://github.com/anjanarprasad/csc786-ethical-data-workflow.git'
ROOT = Path('/content') / REPO_NAME

if not (ROOT / 'README.md').is_file():
    if ROOT.exists():
        raise FileExistsError(f'Repository folder exists but is incomplete: {ROOT}')
    subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)

folder = input('Dataset folder in your Google Drive (Enter for data/raw): ').strip()
if folder:
    from google.colab import drive
    drive.mount('/content/drive')
    my_drive = Path('/content/drive/MyDrive').resolve()
    RAW_DIR = (my_drive / folder).resolve()
    if not RAW_DIR.is_relative_to(my_drive):
        raise ValueError('Enter a folder within your own My Drive.')
    INPUT_STORAGE = 'Google Drive'
else:
    RAW_DIR = ROOT / 'data' / 'raw'
    INPUT_STORAGE = 'Repository data/raw'

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f'Dataset folder not found: {RAW_DIR}')

SOURCES = {
    'NF-CSE-CIC-IDS2018-v2': 'https://rdm.uq.edu.au/files/ce5161d0-ef9c-11ed-827d-e762de186848',
    'NF-UNSW-NB15-v2': 'https://rdm.uq.edu.au/files/8c6e2a00-ef9c-11ed-827d-e762de186848',
}
PUBLISHED_ROW_COUNTS = {
    'NF-CSE-CIC-IDS2018-v2': 18_893_708,
    'NF-UNSW-NB15-v2': 2_390_275,
}
SEED = 42
CHUNK_SIZE = 50_000
SAMPLE_PER_CLASS = 2_000
DOWNLOAD_DATES = {name: None for name in SOURCES}

OUTPUT_DIR = ROOT / 'data' / 'processed'
PROVENANCE_DIR = ROOT / 'data' / 'provenance'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)

print('Input:', INPUT_STORAGE)
print('Python:', platform.python_version(), 'pandas:', pd.__version__, 'numpy:', np.__version__)

Dataset folder in your Google Drive (Enter for data/raw): datanew
Mounted at /content/drive
Input: Google Drive
Python: 3.13.18 pandas: 2.2.3 numpy: 2.1.3


## 2. Find the CSV files

The notebook looks for each dataset by filename, including files in subfolders. Keep one CSV per dataset.

In [2]:
def normalize_filename(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

csv_files = sorted(RAW_DIR.rglob('*.csv'))
matched = {}
missing = []
for name in SOURCES:
    matches = [path for path in csv_files
               if normalize_filename(name) in normalize_filename(path.stem)]
    if len(matches) > 1:
        raise ValueError(f'Multiple CSV files match {name}: {matches}')
    if matches:
        matched[name] = matches[0]
    else:
        missing.append(name)

print('Found datasets:')
for name, path in matched.items():
    print(f'  {name}: {path.relative_to(RAW_DIR)} ({path.stat().st_size / 1024**2:.1f} MB)')
if missing:
    print('Missing:', ', '.join(missing))
if not matched:
    raise FileNotFoundError(f'No matching CSV files found in {RAW_DIR}. Check the Drive folder and filenames.')

Found datasets:
  NF-CSE-CIC-IDS2018-v2: NF-CSE-CIC-IDS2018-v2.csv (3072.1 MB)
  NF-UNSW-NB15-v2: NF-UNSW-NB15-v2.csv (421.4 MB)


## 3. Validate and sample

Read the CSVs in chunks, check labels, and sample up to 2,000 rows per class using fixed seeds. Rows with missing labels are counted and excluded from sampling.

In [3]:
SENSITIVE_COLUMNS = {
    "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "IPV6_SRC_ADDR", "IPV6_DST_ADDR",
    "SRC_IP", "DST_IP", "SOURCE_IP", "DESTINATION_IP", "FLOW_ID",
}
LABEL_VALUES = {"0": 0, "1": 1, "0.0": 0, "1.0": 1,
                "benign": 0, "normal": 0, "malicious": 1, "attack": 1}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def binary_labels(values):
    normalized = values.astype("string").str.strip().str.lower()
    return normalized.map(LABEL_VALUES)

def validate_and_sample(dataset_name, csv_path, seed):
    rng = np.random.default_rng(seed)
    samples = {0: pd.DataFrame(), 1: pd.DataFrame()}
    class_counts = {0: 0, 1: 0}
    missing_by_column = None
    original_columns = None
    total_rows = 0
    invalid_labels = 0
    missing_label_rows = 0
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, low_memory=False):
        chunk.columns = [str(name).strip() for name in chunk.columns]
        if original_columns is None:
            original_columns = list(chunk.columns)
            label_matches = [name for name in chunk.columns if name.lower() == "label"]
            if len(label_matches) != 1:
                raise ValueError(f"{dataset_name}: expected one binary 'Label' column, got {label_matches}")
            label_column = label_matches[0]
            missing_by_column = pd.Series(0, index=original_columns, dtype="int64")
        elif list(chunk.columns) != original_columns:
            raise ValueError(f"{dataset_name}: inconsistent CSV columns")
        total_rows += len(chunk)
        missing_by_column = missing_by_column.add(chunk.isna().sum(), fill_value=0).astype("int64")
        labels = binary_labels(chunk[label_column])
        raw_labels = chunk[label_column].astype("string").str.strip()
        missing_label_mask = raw_labels.isna() | raw_labels.eq("").fillna(False)
        missing_label_rows += int(missing_label_mask.sum())
        invalid_labels += int((labels.isna() & ~missing_label_mask).sum())
        priorities = rng.random(len(chunk))
        for label in (0, 1):
            row_mask = labels.eq(label).fillna(False).to_numpy(dtype=bool)
            class_counts[label] += int(row_mask.sum())
            if not row_mask.any():
                continue
            frame = chunk.loc[row_mask].copy()
            frame["binary_label"] = label
            frame["__sample_priority"] = priorities[row_mask]
            combined = pd.concat([samples[label], frame], ignore_index=True)
            samples[label] = combined.nsmallest(SAMPLE_PER_CLASS, "__sample_priority")
    if total_rows == 0:
        raise ValueError(f"{dataset_name}: empty CSV")
    if invalid_labels:
        raise ValueError(f"{dataset_name}: {invalid_labels} labels not recognized as binary 0/1")
    selected = pd.concat([samples[0], samples[1]], ignore_index=True)
    selected = selected.sort_values("__sample_priority").drop(columns=["__sample_priority"])
    selected["source_dataset"] = dataset_name
    removed = [c for c in selected.columns if c.upper().replace(" ", "_") in SENSITIVE_COLUMNS]
    selected = selected.drop(columns=removed)
    selected = selected.reset_index(drop=True)
    info = {
        "records_in_original_csv": int(total_rows),
        "rows_excluded_missing_label": int(missing_label_rows),
        "original_column_count": len(original_columns),
        "columns_in_original_csv": original_columns,
        "class_counts_full_data": {str(k): v for k, v in class_counts.items()},
        "missing_values_full_data": {col: int(count) for col, count in missing_by_column.items() if count > 0},
        "sample_rows": int(len(selected)),
        "sample_class_counts": {str(k): int((selected['binary_label'] == k).sum()) for k in (0, 1)},
        "duplicate_rows_in_sample": int(selected.duplicated().sum()),
        "excluded_columns_in_sample": removed,
    }
    return selected, info

## 4. Save results

Save samples, checksums, and run details. The source CSVs are not modified.

In [4]:
run_summaries = []
for name, file_path in matched.items():
    dataset_seed = SEED + list(SOURCES).index(name)
    print("\nProcessing:", name)
    sample, validation = validate_and_sample(name, file_path, seed=dataset_seed)
    source_sha256 = sha256_file(file_path)
    stem = name.lower()
    output_csv = OUTPUT_DIR / f"{stem}_sample.csv"
    sample.to_csv(output_csv, index=False)
    generated_sha256 = sha256_file(output_csv)
    run = {
        "dataset": name,
        "official_source": SOURCES[name],
        "relative_input_path": str(file_path.relative_to(RAW_DIR)),
        "input_storage": INPUT_STORAGE,
        "download_date_utc_user_recorded": DOWNLOAD_DATES[name],
        "processing_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source_size_bytes": int(file_path.stat().st_size),
        "source_sha256": source_sha256,
        "output_relative_path": str(output_csv.relative_to(ROOT)),
        "output_sha256": generated_sha256,
        "parameters": {"random_seed": dataset_seed, "chunk_size": CHUNK_SIZE,
                       "max_sample_per_class": SAMPLE_PER_CLASS},
        "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__},
        "validation": validation,
        "status": "completed",
    }
    manifest = PROVENANCE_DIR / f"{stem}_provenance.json"
    run_summaries.append({
        "dataset": name,
        "processed_utc": run["processing_timestamp_utc"],
        "download_date_utc": run["download_date_utc_user_recorded"],
        "records_loaded": validation["records_in_original_csv"],
        "rows_excluded_missing_label": validation["rows_excluded_missing_label"],
        "sample_rows": validation["sample_rows"],
        "source_sha256": source_sha256,
        "sample_sha256": generated_sha256,
        "status": "completed",
    })
    published_total = PUBLISHED_ROW_COUNTS[name]
    count_matches_published = validation["records_in_original_csv"] == published_total
    run["published_row_count"] = published_total
    run["source_row_count_matches_published"] = count_matches_published
    manifest.write_text(json.dumps(run, indent=2), encoding="utf-8")
    if not count_matches_published:
        print(f"WARNING: {name} contains {validation['records_in_original_csv']:,} rows; "
              f"UQ publishes {published_total:,}. Verify the source file and completeness.")
    print("Records:", validation["records_in_original_csv"],
          "Binary classes:", validation["class_counts_full_data"])
    print("Excluded rows with missing/blank labels:", validation["rows_excluded_missing_label"])
    print("Missing-value columns:", len(validation["missing_values_full_data"]),
          "Sample duplicate rows:", validation["duplicate_rows_in_sample"])
    print("Saved sample:", output_csv.relative_to(ROOT))
    print("Source SHA-256:", source_sha256)
    print("Sample SHA-256:", generated_sha256)

run_log = ROOT / "data" / "collection_runs.csv"
new_runs = pd.DataFrame(run_summaries)
if run_log.exists() and run_log.stat().st_size > 0:
    try:
        previous = pd.read_csv(run_log)
        new_runs = pd.concat([previous, new_runs], ignore_index=True)
    except pd.errors.EmptyDataError:
        print("The existing collection log was empty; writing a new log.")
new_runs.to_csv(run_log, index=False)
print("\nSaved collection log:", run_log.relative_to(ROOT))
print("Processed datasets:", len(run_summaries), "of", len(SOURCES))
print("Missing datasets:", missing)
display(pd.DataFrame(run_summaries)[["dataset", "records_loaded", "sample_rows", "status"]])


Processing: NF-CSE-CIC-IDS2018-v2
Records: 18893708 Binary classes: {'0': 16635567, '1': 2258141}
Excluded rows with missing/blank labels: 0
Missing-value columns: 0 Sample duplicate rows: 2
Saved sample: data/processed/nf-cse-cic-ids2018-v2_sample.csv
Source SHA-256: 16a545a3e139b2e85d08b1b7d9d40328c3689a49d0deaafce6dcf8d7b63e1257
Sample SHA-256: f281d0b3f622d8f76dbeae69c7d13cf1979d3fa99e0d54eaffbd95d7038e571a

Processing: NF-UNSW-NB15-v2
Records: 2390275 Binary classes: {'0': 2295222, '1': 95053}
Excluded rows with missing/blank labels: 0
Missing-value columns: 0 Sample duplicate rows: 21
Saved sample: data/processed/nf-unsw-nb15-v2_sample.csv
Source SHA-256: 05019e3ac8d55b3f074f9f042623bbbb1af2527b8b78558c69353ccb2ccefa3f
Sample SHA-256: 7125d0da4ed69abb11823fa9ce4be94db6a708c5c73a02c05080c470408027b0

Saved collection log: data/collection_runs.csv
Processed datasets: 2 of 2
Missing datasets: []


,dataset,records_loaded,sample_rows,status
0,NF-CSE-CIC-IDS2018-v2,18893708,4000,completed
1,NF-UNSW-NB15-v2,2390275,4000,completed


## 5. Verify samples

Check the labels, excluded columns, and saved file hashes.

In [5]:
for item in run_summaries:
    sample_path = OUTPUT_DIR / f"{item['dataset'].lower()}_sample.csv"
    sample_df = pd.read_csv(sample_path)
    assert "binary_label" in sample_df.columns
    assert set(sample_df["binary_label"].dropna().unique()).issubset({0, 1})
    assert len(sample_df) <= 2 * SAMPLE_PER_CLASS
    assert sha256_file(sample_path) == item["sample_sha256"]
    assert not any(c.upper().replace(" ", "_") in SENSITIVE_COLUMNS for c in sample_df.columns)
    print(f"Verified {item['dataset']}: {len(sample_df)} sampled rows; file checksum matches")
print("Validation finished. Do not commit the files inside data/raw, data/processed, data/provenance, or data/api.")

Verified NF-CSE-CIC-IDS2018-v2: 4000 sampled rows; file checksum matches
Verified NF-UNSW-NB15-v2: 4000 sampled rows; file checksum matches
Validation finished. Do not commit the files inside data/raw, data/processed, data/provenance, or data/api.


## 6. NVD API

To run this section, request your own NVD API key at https://nvd.nist.gov/developers/request-an-api-key and save it as `NVD_API_KEY` in Colab Secrets. Allow notebook access. The code retrieves `CVE-2021-44228` and saves the public record and its provenance under `data/api/`.

The CVE record is not part of the NIDS datasets. This product uses the NVD API but is not endorsed or certified by the NVD.

In [6]:
import requests

NVD_ENDPOINT = "https://services.nvd.nist.gov/rest/json/cves/2.0"
NVD_PARAMS = {"cveId": "CVE-2021-44228"}
NVD_API_STATUS = "not_started"

nvd_api_key = None
try:
    from google.colab import userdata
    nvd_api_key = userdata.get("NVD_API_KEY")
except Exception:
    nvd_api_key = os.environ.get("NVD_API_KEY")
nvd_api_key = nvd_api_key.strip() if nvd_api_key else None

if not nvd_api_key:
    NVD_API_STATUS = "skipped_missing_key"
    print("NVD API: SKIPPED (add NVD_API_KEY to Colab Secrets and enable notebook access).")
else:
    try:
        response = requests.get(
            NVD_ENDPOINT,
            params=NVD_PARAMS,
            headers={"apiKey": nvd_api_key},
            timeout=30,
        )
        response.raise_for_status()
        response_bytes = response.content
        payload = response.json()
        entries = payload.get("vulnerabilities", [])
        if len(entries) != 1 or entries[0].get("cve", {}).get("id") != NVD_PARAMS["cveId"]:
            raise ValueError("The NVD response does not contain the requested single CVE")

        cve = entries[0]["cve"]
        description = next((item.get("value", "") for item in cve.get("descriptions", [])
                            if item.get("lang") == "en"), "")
        record = {
            "cve_id": cve["id"],
            "published": cve.get("published", ""),
            "last_modified": cve.get("lastModified", ""),
            "description_en": description,
        }

        api_dir = ROOT / "data" / "api"
        api_dir.mkdir(parents=True, exist_ok=True)
        output_csv = api_dir / "nvd_cve_example.csv"
        output_json = api_dir / "nvd_cve_provenance.json"
        pd.DataFrame([record]).to_csv(output_csv, index=False)

        provenance = {
            "provider": "NIST National Vulnerability Database",
            "endpoint": NVD_ENDPOINT,
            "parameters": NVD_PARAMS,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "authentication": "API key supplied in request header; not stored",
            "record_count": 1,
            "raw_response_sha256": hashlib.sha256(response_bytes).hexdigest(),
            "output_csv_sha256": sha256_file(output_csv),
            "output_relative_path": str(output_csv.relative_to(ROOT)),
            "python_version": platform.python_version(),
            "pandas_version": pd.__version__,
            "status": "completed",
        }
        output_json.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
        NVD_API_STATUS = "completed"
        print("NVD API: SUCCESS")
        print("CVE retrieved:", cve["id"])
        print("Records collected: 1")
        print("CSV saved:", output_csv.relative_to(ROOT))
        print("Provenance saved:", output_json.relative_to(ROOT))
        print("CSV SHA-256:", provenance["output_csv_sha256"])
    except requests.exceptions.RequestException as exc:
        NVD_API_STATUS = "failed"
        print("NVD API: FAILED (HTTP or network error).", type(exc).__name__)
    except (ValueError, KeyError, TypeError) as exc:
        NVD_API_STATUS = "failed"
        print("NVD API: FAILED (response validation).", type(exc).__name__)
    finally:
        nvd_api_key = None

NVD API: SUCCESS
CVE retrieved: CVE-2021-44228
Records collected: 1
CSV saved: data/api/nvd_cve_example.csv
Provenance saved: data/api/nvd_cve_provenance.json
CSV SHA-256: fc744b4ce7adadaf187f38e7103b2fe160edb0b5a472f0497134a1f875764be8
